In [ ]:
# Montamos Google Drive para acceder al dataset y a los ficheros auxiliares
# que se van generando durante todo el pipeline (CSV intermedios, modelos, etc.)
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

In [ ]:
# Nos situamos en la carpeta de trabajo del proyecto dentro de Drive,
# donde estan los 75 CSV originales del dataset BoT-IoT
%cd '/content/drive/MyDrive/TFG/archive'

In [ ]:
import pandas as pd

# 1. Cargamos el fichero auxiliar que contiene los nombres reales de las columnas
names_df = pd.read_csv("data_names.csv")

# 2. Mostramos las columnas para verificar que el fichero se ha leido bien
print(names_df.columns.tolist())

**Combino los 75 csv**

In [ ]:
import pandas as pd
import glob
import os
from pathlib import Path

# --- Combinacion incremental de los 75 CSV originales ---
# El volumen total de datos es demasiado grande para cargarlo en memoria de
# una sola vez, asi que procesamos cada fichero por bloques (chunks) de 50.000
# filas y vamos escribiendo el resultado limpio de forma incremental.

output_file = 'bot_iot_limpio_incremental.csv'
first_write = not os.path.exists(output_file)
mode = 'w' if first_write else 'a'  # Append despues del primer fichero
header = first_write

# Localizamos todos los ficheros data_*.csv del dataset original
csv_files = sorted(glob.glob("data_*.csv"))
print(f" {len(csv_files)} archivos")

total_rows = 0
for i, file in enumerate(csv_files, 1):
    print(f"\n[{i}/{len(csv_files)}] {os.path.basename(file)}")

    # Leemos el fichero por bloques para no agotar la memoria RAM
    chunk_iter = pd.read_csv(file, header=None, chunksize=50000, low_memory=False)

    for chunk_num, df_temp in enumerate(chunk_iter):
        print(f"  Chunk {chunk_num+1}...")
        n_cols = df_temp.shape[1]

        # Algunos CSV originales tienen columnas duplicadas o adicionales,
        # asi que generamos nombres repitiendo la plantilla base hasta cubrir
        # el numero real de columnas de este fichero concreto
        base_cols = ['pkSeqID', 'stime', 'flgs', 'proto', 'saddr', 'sport', 'daddr', 'dport',
                     'pkts', 'bytes', 'spkts', 'dpkts', 'sbytes', 'dbytes', 'rate', 'srate',
                     'drate', 'attack', 'category', 'subcategory']
        col_names = (base_cols * 3)[:n_cols]

        # Nos aseguramos de que los nombres sean únicos (si se repiten,
        # añadimos un sufijo numerico) para evitar columnas duplicadas
        seen = {}
        final_names = []
        for col in col_names:
            name = col if col not in seen else f"{col}_{seen[col]}"
            final_names.append(name)
            seen[col] = seen.get(col, 0) + 1

        df_temp.columns = final_names

        # Si la primera fila del chunk es en realidad una repeticion del
        # encabezado (en vez de un dato real), la eliminamos
        if len(df_temp) > 0 and df_temp.iloc[0, 0] == 'pkSeqID':
            df_temp = df_temp.iloc[1:].reset_index(drop=True)

        # Convertimos a numerico las columnas estadisticas del flujo,
        # sustituyendo por 0 cualquier valor no convertible (para evitar que un
        # dato corrupto interrumpa todo el pipeline)
        possible_num = ['pkts', 'bytes', 'spkts', 'dpkts', 'sbytes', 'dbytes', 'rate', 'srate', 'drate']
        for col in possible_num:
            if col in df_temp.columns:
                df_temp[col] = pd.to_numeric(df_temp[col], errors='coerce').fillna(0)

        # Normalizamos la columna 'attack' a un entero limpio (0/1)
        attack_col = next((c for c in df_temp.columns if 'attack' in c.lower()), None)
        if attack_col:
            df_temp['attack_clean'] = pd.to_numeric(df_temp[attack_col], errors='coerce').fillna(0).astype(int)

        # Descartamos filas invalidas: un flujo de red real siempre tiene
        # al menos un paquete (pkts > 0)
        pkts_col = next((c for c in df_temp.columns if 'pkts' in c.lower()), None)
        if pkts_col:
            df_temp = df_temp[df_temp[pkts_col] > 0]

        # Escribimos este chunk ya limpio al fichero de salida acumulado
        df_temp.to_csv(output_file, mode=mode, header=header, index=False)
        total_rows += len(df_temp)
        mode = 'a'
        header = False
        print(f"    -> {len(df_temp):,} validas escritas (total: {total_rows:,})")

**Limpio el csv**

Diagnóstico para aclarar el contenido

In [ ]:
# Diagnostico: leemos el CSV combinado sin asumir que la primera fila es el
# encabezado, para ver exactamente que hay en cada posicion de columna y
# confirmar si la fila 0 es cabecera o ya es un dato real
df_raw = pd.read_csv('bot_iot_limpio_incremental.csv',
                      nrows=5,
                      header=None,  # ignoramos el header por ahora
                      low_memory=False)

print(f"Total columnas: {df_raw.shape[1]}")
print("\nFila 0 (¿es header o dato?):")
for i, val in enumerate(df_raw.iloc[0]):
    print(f"  Col {i:2d}: {repr(val)}")

print("\nFila 1 (primer dato real):")
for i, val in enumerate(df_raw.iloc[1]):
    print(f"  Col {i:2d}: {repr(val)}")

In [ ]:
import pandas as pd
import os

# --- Segunda pasada: seleccion final de columnas y calculo de 'attack' ---
# A partir del diagnostico anterior, ya sabemos en que posicion exacta esta
# cada columna que realmente necesitamos, asi que las seleccionamos por
# indice numerico (usecols) en vez de por nombre, que resulto poco fiable
# por las inconsistencias entre ficheros.

input_file  = 'bot_iot_limpio_incremental.csv'
output_file = 'bot_iot_final_ml.csv'

# Mapeo posicion de columna -> nombre real
col_map = {
    0:  'pkSeqID',
    1:  'stime',
    2:  'flgs',
    3:  'proto',
    4:  'saddr',
    5:  'sport',
    6:  'daddr',
    7:  'dport',
    8:  'pkts',
    9:  'bytes',
    25: 'spkts',
    26: 'dpkts',
    27: 'sbytes',
    28: 'dbytes',
    13: 'rate',
    14: 'srate',
    15: 'drate',
    33: 'category',
    34: 'subcategory'
}

usecols  = sorted(col_map.keys())
num_cols = ['pkts','bytes','spkts','dpkts','sbytes','dbytes','rate','srate','drate']

mode   = 'w'
header = True
total  = 0

# Procesamos de nuevo por bloques (esta vez de 200.000 filas, el fichero
# combinado ya es mas manejable que los 75 originales por separado)
for chunk in pd.read_csv(
        input_file,
        chunksize=200000,
        low_memory=False,
        header=None,
        skiprows=1,
        usecols=usecols):

    chunk.rename(columns=col_map, inplace=True)

    # Eliminamos filas que sean en realidad cabeceras repetidas coladas
    # entre los datos, y filas donde 'pkts' no sea un numero valido
    chunk = chunk[chunk['pkts'].astype(str).str.strip() != 'pkts'].copy()
    chunk = chunk[pd.to_numeric(chunk['pkts'], errors='coerce').notna()].copy()

    # Convertimos las columnas estadisticas a numerico
    for col in num_cols:
        chunk[col] = pd.to_numeric(chunk[col], errors='coerce').fillna(0)

    # Derivamos 'attack' directamente de 'category', que es la fuente de
    # verdad mas fiable: cualquier categoria distinta de 'Normal' es ataque
    chunk['category'] = chunk['category'].astype(str).str.strip()
    chunk['attack']   = (chunk['category'] != 'Normal').astype(int)

    # Filtro basico: descartamos flujos sin paquetes
    chunk = chunk[chunk['pkts'] > 0]

    if len(chunk) == 0:
        continue

    # Reordenamos columnas dejando attack/category/subcategory al final
    out_cols = ['pkSeqID','stime','flgs','proto','saddr','sport','daddr','dport',
                'pkts','bytes','spkts','dpkts','sbytes','dbytes','rate','srate','drate',
                'attack','category','subcategory']
    chunk = chunk[out_cols]

    chunk.to_csv(output_file, mode=mode, header=header, index=False)
    mode   = 'a'
    header = False
    total += len(chunk)
    print(f"  -> {total:,} filas escritas", end='\r')

print(f"\n\nLISTO: {total:,} filas en {output_file}")

# Verificacion final: comprobamos que las proporciones de clase y los
# valores tienen sentido antes de seguir adelante
df_check = pd.read_csv(output_file, nrows=1000000)
print(f"\n Attack:\n{df_check['attack'].value_counts(normalize=True).round(3)}")
print(f"\n Category:\n{df_check['category'].value_counts(normalize=True).round(3).head(10)}")
print(f"\n Subcategory:\n{df_check['subcategory'].value_counts(normalize=True).round(3).head(10)}")
print(f"\n Muestra:\n{df_check[['pkts','attack','category','subcategory']].sample(5)}")

**EDA y preparación para ML**:

El dataset completo tiene un desbalanceo extremo: Normal y Theft representan apenas el 0,01% y 0,002% respectivamente. Si entrenamos directamente sobre los 73M de registros:

El modelo aprenderá casi exclusivamente DDoS/DoS

Theft (1.587 registros) quedaría completamente ignorado



In [ ]:
import pandas as pd
import numpy as np

# --- Muestreo estratificado ---
# El dataset completo es demasiado grande para entrenar modelos de forma
# comoda en Colab, asi que tomamos una muestra proporcional de cada
# categoria de ataque. Las clases minoritarias (Normal, Theft) se toman
# completas; de las clases mayoritarias (DDoS, DoS, Reconnaissance) se
# toman 200.000 filas de cada una.

frames = []
proporciones = {
    'DDoS': 200000,
    'DoS': 200000,
    'Reconnaissance': 200000,
    'Normal': 9787,
    'Theft': 1587
}

# Recorremos el CSV completo por bloques, acumulando muestras de cada
# categoria hasta alcanzar la cantidad objetivo
for chunk in pd.read_csv('bot_iot_final_ml.csv',
                          chunksize=500000,
                          low_memory=False):
    for cat, n_needed in proporciones.items():
        sub = chunk[chunk['category'] == cat]
        if len(sub) > 0 and n_needed > 0:
            n_take = min(len(sub), n_needed)
            frames.append(sub.sample(n=n_take, random_state=42))
            proporciones[cat] -= n_take

    # Si ya hemos completado todas las cuotas, no hace falta seguir leyendo
    if all(v <= 0 for v in proporciones.values()):
        break

# Concatenamos todas las muestras y mezclamos las filas aleatoriamente
df_sample = pd.concat(frames).sample(frac=1, random_state=42).reset_index(drop=True)
print(f"Muestra estratificada: {len(df_sample):,} filas")
print(f"\nDistribucion muestra:\n{df_sample['category'].value_counts()}")

# Estadisticas descriptivas de las features numericas, para tener una
# primera vision del rango y la dispersion de cada variable
num_cols = ['pkts','bytes','spkts','dpkts','sbytes','dbytes','rate','srate','drate']
print(f"\n Estadisticas descriptivas:\n{df_sample[num_cols].describe().round(3)}")

# Guardamos la muestra final, que sera la base para entrenar todos los modelos
df_sample.to_csv('bot_iot_sample_ml.csv', index=False)
print("\n Muestra guardada en bot_iot_sample_ml.csv")

**Preparación del entrenamiento:**

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, MinMaxScaler
from sklearn.metrics import classification_report, accuracy_score, f1_score
from imblearn.over_sampling import SMOTE
import warnings
warnings.filterwarnings('ignore')

# Cargamos la muestra estratificada que preparamos en el paso anterior
df = pd.read_csv('bot_iot_sample_ml.csv', low_memory=False)

# --- Preparacion de features y etiquetas ---
# Usamos solo las 9 variables estadisticas de flujo, evitando a proposito
# direcciones IP, puertos y otros identificadores que podrian provocar
# sobreajuste sin aportar capacidad real de generalizacion
features = ['pkts','bytes','spkts','dpkts','sbytes','dbytes','rate','srate','drate']
X = df[features].copy()
y_bin   = df['attack'].astype(int)

# Codificamos la etiqueta multiclase (category) como enteros
le = LabelEncoder()
y_multi = le.fit_transform(df['category'])
print(f"Clases: {le.classes_}")

# Normalizacion Min-Max: escalamos todas las variables al rango [0,1]
# para que ningun atributo domine el aprendizaje solo por tener valores
# numericamente mas grandes
scaler  = MinMaxScaler()
X_scaled = scaler.fit_transform(X)

# Division 80/20 estratificada: mantenemos la misma proporcion de clases
# en train y en test que en el dataset original
X_tr, X_te, yb_tr, yb_te, ym_tr, ym_te = train_test_split(
    X_scaled, y_bin, y_multi,
    test_size=0.2, random_state=42, stratify=y_bin)

print(f"Train: {len(X_tr):,} | Test: {len(X_te):,}")

# --- Balanceo con SMOTE ---
# Aplicamos SMOTE UNICAMENTE sobre el conjunto de entrenamiento, generando
# muestras sinteticas de las clases minoritarias por interpolacion entre
# vecinos cercanos. El conjunto de test se deja intacto para que la
# evaluacion final sea realista y no este contaminada por datos sinteticos.
smote = SMOTE(random_state=42)
X_tr_b, yb_tr_s = smote.fit_resample(X_tr, yb_tr)
X_tr_m, ym_tr_s = smote.fit_resample(X_tr, ym_tr)
print(f"Train SMOTE binaria:    {len(X_tr_b):,}")
print(f"Train SMOTE multiclase: {len(X_tr_m):,}")

**Entrenamiento con Random Forest**

In [ ]:
from sklearn.ensemble import RandomForestClassifier

print("=" * 55)
print("RANDOM FOREST")
print("=" * 55)

# --- Clasificacion BINARIA (ataque vs normal) ---
rf_bin = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
rf_bin.fit(X_tr_b, yb_tr_s)
yb_pred_rf = rf_bin.predict(X_te)

print("\n-- Clasificacion BINARIA --")
print(f"Accuracy : {accuracy_score(yb_te, yb_pred_rf)*100:.2f}%")
print(f"F1-Score : {f1_score(yb_te, yb_pred_rf, average='weighted')*100:.2f}%")
print(classification_report(yb_te, yb_pred_rf, target_names=['Normal','Attack']))

# --- Clasificacion MULTICLASE (tipo concreto de ataque) ---
rf_multi = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
rf_multi.fit(X_tr_m, ym_tr_s)
ym_pred_rf = rf_multi.predict(X_te)

print("\n-- Clasificacion MULTICLASE --")
print(f"Accuracy : {accuracy_score(ym_te, ym_pred_rf)*100:.2f}%")
print(f"F1-Score : {f1_score(ym_te, ym_pred_rf, average='weighted')*100:.2f}%")
print(classification_report(ym_te, ym_pred_rf, target_names=le.classes_))

# Importancia de cada caracteristica segun el modelo multiclase, util
# para entender que variables pesan mas en la decision del arbol
fi_rf = pd.DataFrame({'feature': features,
                       'importance': rf_multi.feature_importances_}
                     ).sort_values('importance', ascending=False)
print(f"\n Feature Importance RF:\n{fi_rf.to_string(index=False)}")

**Entrenamiento con XGBoost**

In [ ]:
from xgboost import XGBClassifier

print("=" * 55)
print("XGBOOST")
print("=" * 55)

# --- Clasificacion BINARIA ---
xgb_bin = XGBClassifier(n_estimators=100, random_state=42,
                         eval_metric='logloss', n_jobs=-1, verbosity=0)
xgb_bin.fit(X_tr_b, yb_tr_s)
yb_pred_xgb = xgb_bin.predict(X_te)

print("\n-- Clasificacion BINARIA --")
print(f"Accuracy : {accuracy_score(yb_te, yb_pred_xgb)*100:.2f}%")
print(f"F1-Score : {f1_score(yb_te, yb_pred_xgb, average='weighted')*100:.2f}%")
print(classification_report(yb_te, yb_pred_xgb, target_names=['Normal','Attack']))

# --- Clasificacion MULTICLASE ---
xgb_multi = XGBClassifier(n_estimators=100, random_state=42,
                            objective='multi:softmax',
                            num_class=len(le.classes_),
                            eval_metric='mlogloss', n_jobs=-1, verbosity=0)
xgb_multi.fit(X_tr_m, ym_tr_s)
ym_pred_xgb = xgb_multi.predict(X_te)

print("\n-- Clasificacion MULTICLASE --")
print(f"Accuracy : {accuracy_score(ym_te, ym_pred_xgb)*100:.2f}%")
print(f"F1-Score : {f1_score(ym_te, ym_pred_xgb, average='weighted')*100:.2f}%")
print(classification_report(ym_te, ym_pred_xgb, target_names=le.classes_))

# Importancia de caracteristicas segun XGBoost (para comparar mas
# adelante con la de Random Forest, ya que ambos modelos discrepan
# notablemente en el peso que asignan a dpkts)
fi_xgb = pd.DataFrame({'feature': features,
                        'importance': xgb_multi.feature_importances_}
                      ).sort_values('importance', ascending=False)
print(f"\n Feature Importance XGB:\n{fi_xgb.to_string(index=False)}")

**Entrenamiento con SVM(LinearSVC)**

In [ ]:
from sklearn.svm import LinearSVC
from sklearn.calibration import CalibratedClassifierCV

print("=" * 55)
print("SVM (LinearSVC)")
print("=" * 55)

# Usamos LinearSVC en vez del SVC estandar por eficiencia computacional
# con este volumen de datos, manteniendo una precision comparable

# --- Clasificacion BINARIA ---
svm_bin = LinearSVC(max_iter=2000, random_state=42)
svm_bin.fit(X_tr_b, yb_tr_s)
yb_pred_svm = svm_bin.predict(X_te)

print("\n-- Clasificacion BINARIA --")
print(f"Accuracy : {accuracy_score(yb_te, yb_pred_svm)*100:.2f}%")
print(f"F1-Score : {f1_score(yb_te, yb_pred_svm, average='weighted')*100:.2f}%")
print(classification_report(yb_te, yb_pred_svm, target_names=['Normal','Attack']))

# --- Clasificacion MULTICLASE ---
svm_multi = LinearSVC(max_iter=2000, random_state=42)
svm_multi.fit(X_tr_m, ym_tr_s)
ym_pred_svm = svm_multi.predict(X_te)

print("\n-- Clasificacion MULTICLASE --")
print(f"Accuracy : {accuracy_score(ym_te, ym_pred_svm)*100:.2f}%")
print(f"F1-Score : {f1_score(ym_te, ym_pred_svm, average='weighted')*100:.2f}%")
print(classification_report(ym_te, ym_pred_svm, target_names=le.classes_))

**Entrenamienyo con Autoenconder + XGBoost Binario**


In [ ]:
import numpy as np
import tensorflow as tf
from tensorflow.keras import Model
from tensorflow.keras.layers import Input, Dense
from tensorflow.keras.callbacks import EarlyStopping
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, f1_score, classification_report

tf.random.set_seed(42)
np.random.seed(42)

print("=" * 55)
print("AUTOENCODER + XGBOOST")
print("=" * 55)

# --- Propuesta hibrida: autoencoder como extractor de caracteristicas ---
# El autoencoder comprime las 9 variables originales en un espacio latente
# de menor dimension; despues, XGBoost se entrena sobre esa representacion
# comprimida en vez de sobre las variables originales.

# =========================
# VERSION BINARIA
# =========================
input_dim_b = X_tr_b.shape[1]
latent_dim_b = 6

# Arquitectura simetrica: codificador (comprime) + decodificador (reconstruye)
inp_b = Input(shape=(input_dim_b,))
x = Dense(32, activation='relu')(inp_b)
x = Dense(16, activation='relu')(x)
latent_b = Dense(latent_dim_b, activation='relu', name='latent_b')(x)
x = Dense(16, activation='relu')(latent_b)
x = Dense(32, activation='relu')(x)
out_b = Dense(input_dim_b, activation='sigmoid')(x)

autoencoder_b = Model(inp_b, out_b)
# El "encoder" es el mismo modelo pero cortado justo en la capa latente,
# lo usaremos despues para proyectar datos nuevos a ese espacio comprimido
encoder_b = Model(inp_b, latent_b)

autoencoder_b.compile(optimizer='adam', loss='mse')

# Parada temprana para evitar sobreajuste del autoencoder
early_stop = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)

autoencoder_b.fit(
    X_tr_b, X_tr_b,  # el autoencoder aprende a reconstruir su propia entrada
    epochs=30,
    batch_size=256,
    validation_split=0.1,
    callbacks=[early_stop],
    verbose=1
)

# Proyectamos train y test al espacio latente aprendido
X_tr_b_ae = encoder_b.predict(X_tr_b, verbose=0)
X_te_b_ae = encoder_b.predict(X_te, verbose=0)

# XGBoost se entrena ahora sobre las 6 variables latentes, no sobre las 9 originales
xgb_ae_bin = XGBClassifier(
    n_estimators=200,
    max_depth=6,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    eval_metric='logloss',
    random_state=42
)

xgb_ae_bin.fit(X_tr_b_ae, yb_tr_s)
yb_pred_xgb_ae = xgb_ae_bin.predict(X_te_b_ae)

print("\n-- Clasificacion BINARIA --")
print(f"Accuracy : {accuracy_score(yb_te, yb_pred_xgb_ae)*100:.2f}%")
print(f"F1-Score : {f1_score(yb_te, yb_pred_xgb_ae, average='weighted')*100:.2f}%")
print(classification_report(yb_te, yb_pred_xgb_ae, target_names=['Normal','Attack']))

**Multiclase**

In [ ]:
# =========================
# VERSION MULTICLASE (misma arquitectura que la binaria)
# =========================
input_dim_m = X_tr_m.shape[1]
latent_dim_m = 6

inp_m = Input(shape=(input_dim_m,))
x = Dense(32, activation='relu')(inp_m)
x = Dense(16, activation='relu')(x)
latent_m = Dense(latent_dim_m, activation='relu', name='latent_m')(x)
x = Dense(16, activation='relu')(latent_m)
x = Dense(32, activation='relu')(x)
out_m = Dense(input_dim_m, activation='sigmoid')(x)

autoencoder_m = Model(inp_m, out_m)
encoder_m = Model(inp_m, latent_m)

autoencoder_m.compile(optimizer='adam', loss='mse')

autoencoder_m.fit(
    X_tr_m, X_tr_m,
    epochs=30,
    batch_size=256,
    validation_split=0.1,
    callbacks=[early_stop],
    verbose=1
)

# Proyectamos al espacio latente y entrenamos XGBoost multiclase sobre el
X_tr_m_ae = encoder_m.predict(X_tr_m, verbose=0)
X_te_m_ae = encoder_m.predict(X_te, verbose=0)

xgb_ae_multi = XGBClassifier(
    n_estimators=200,
    max_depth=6,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    eval_metric='mlogloss',
    random_state=42
)

xgb_ae_multi.fit(X_tr_m_ae, ym_tr_s)
ym_pred_xgb_ae = xgb_ae_multi.predict(X_te_m_ae)

print("\n-- Clasificacion MULTICLASE --")
print(f"Accuracy : {accuracy_score(ym_te, ym_pred_xgb_ae)*100:.2f}%")
print(f"F1-Score : {f1_score(ym_te, ym_pred_xgb_ae, average='weighted')*100:.2f}%")
print(classification_report(ym_te, ym_pred_xgb_ae, target_names=le.classes_))

**Tabla resumen comparativa**

In [ ]:
# --- Tabla resumen comparativa ---
# Recogemos accuracy y F1-score de los 4 modelos, en sus dos tareas
# (binaria y multiclase), en una unica tabla para comparar de un vistazo
resultados = {
    'Modelo': [
        'Random Forest', 'Random Forest',
        'XGBoost', 'XGBoost',
        'SVM', 'SVM',
        'AE + XGBoost', 'AE + XGBoost'
    ],
    'Tipo': [
        'Binaria', 'Multiclase',
        'Binaria', 'Multiclase',
        'Binaria', 'Multiclase',
        'Binaria', 'Multiclase'
    ],
    'Accuracy': [
        accuracy_score(yb_te, yb_pred_rf)*100,
        accuracy_score(ym_te, ym_pred_rf)*100,
        accuracy_score(yb_te, yb_pred_xgb)*100,
        accuracy_score(ym_te, ym_pred_xgb)*100,
        accuracy_score(yb_te, yb_pred_svm)*100,
        accuracy_score(ym_te, ym_pred_svm)*100,
        accuracy_score(yb_te, yb_pred_xgb_ae)*100,
        accuracy_score(ym_te, ym_pred_xgb_ae)*100,
    ],
    'F1-Score': [
        f1_score(yb_te, yb_pred_rf, average='weighted')*100,
        f1_score(ym_te, ym_pred_rf, average='weighted')*100,
        f1_score(yb_te, yb_pred_xgb, average='weighted')*100,
        f1_score(ym_te, ym_pred_xgb, average='weighted')*100,
        f1_score(yb_te, yb_pred_svm, average='weighted')*100,
        f1_score(ym_te, ym_pred_svm, average='weighted')*100,
        f1_score(yb_te, yb_pred_xgb_ae, average='weighted')*100,
        f1_score(ym_te, ym_pred_xgb_ae, average='weighted')*100,
    ]
}

df_res = pd.DataFrame(resultados)
df_res['Accuracy'] = df_res['Accuracy'].round(2)
df_res['F1-Score'] = df_res['F1-Score'].round(2)

print("\n" + "=" * 55)
print(" TABLA COMPARATIVA FINAL")
print("=" * 55)
print(df_res.to_string(index=False))

# Exportamos a CSV para trazabilidad de los resultados experimentales
df_res.to_csv('resultados_modelos.csv', index=False)

In [ ]:
import joblib
import os

# --- Exportacion de todos los modelos entrenados ---
# Guardamos scaler, label encoder y los 4 modelos (en sus versiones
# binaria y multiclase) para poder reutilizarlos despues sin reentrenar,
# en particular para la validacion con el honeypot (ver el notebook
# datasetpropio.ipynb)
os.makedirs('modelos_exportados', exist_ok=True)

joblib.dump(scaler, 'modelos_exportados/scaler.pkl')
joblib.dump(le, 'modelos_exportados/label_encoder.pkl')

joblib.dump(rf_bin,   'modelos_exportados/rf_bin.pkl')
joblib.dump(rf_multi, 'modelos_exportados/rf_multi.pkl')
joblib.dump(xgb_bin,   'modelos_exportados/xgb_bin.pkl')
joblib.dump(xgb_multi, 'modelos_exportados/xgb_multi.pkl')
joblib.dump(svm_bin,   'modelos_exportados/svm_bin.pkl')
joblib.dump(svm_multi, 'modelos_exportados/svm_multi.pkl')

# Los modelos de Keras (autoencoders) se guardan con su propio metodo .save()
encoder_b.save('modelos_exportados/encoder_b.h5')
encoder_m.save('modelos_exportados/encoder_m.h5')
joblib.dump(xgb_ae_bin,   'modelos_exportados/xgb_ae_bin.pkl')
joblib.dump(xgb_ae_multi, 'modelos_exportados/xgb_ae_multi.pkl')

print("Modelos guardados en", os.getcwd() + '/modelos_exportados')

In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

# Funcion auxiliar para generar y guardar una matriz de confusion como imagen
def plot_and_save_confusion(y_true, y_pred, labels, title, filename):
    cm = confusion_matrix(y_true, y_pred)
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=labels)
    fig, ax = plt.subplots(figsize=(6, 5))
    disp.plot(ax=ax, cmap='Blues', values_format='d', colorbar=False)
    plt.title(title)
    plt.xticks(rotation=45, ha='right')
    plt.tight_layout()
    plt.savefig(filename, dpi=200, bbox_inches='tight')
    plt.show()

# Generamos las matrices de confusion multiclase para los tres modelos
# de referencia (no se incluye AE+XGBoost porque su rendimiento ya se
# comenta aparte, y para no saturar la memoria con 8 graficas similares)

# Random Forest - multiclase
plot_and_save_confusion(ym_te, ym_pred_rf, le.classes_,
                          'Matriz de confusion - Random Forest (multiclase)',
                          'cm_rf_multiclase.png')

# XGBoost - multiclase
plot_and_save_confusion(ym_te, ym_pred_xgb, le.classes_,
                          'Matriz de confusion - XGBoost (multiclase)',
                          'cm_xgb_multiclase.png')

# SVM - multiclase
plot_and_save_confusion(ym_te, ym_pred_svm, le.classes_,
                          'Matriz de confusion - SVM (multiclase)',
                          'cm_svm_multiclase.png')

In [ ]:
import pandas as pd
from sklearn.metrics import confusion_matrix

# Version en texto de las mismas matrices de confusion de la celda
# anterior, util para pegar los numeros exactos directamente en la
# memoria del TFG sin depender de leer una imagen
def print_confusion_table(y_true, y_pred, labels, title):
    cm = confusion_matrix(y_true, y_pred)
    df_cm = pd.DataFrame(cm, index=labels, columns=labels)
    print(f"\n=== {title} ===")
    print(df_cm.to_string())

print_confusion_table(ym_te, ym_pred_rf, le.classes_, "Matriz de confusion - Random Forest")
print_confusion_table(ym_te, ym_pred_xgb, le.classes_, "Matriz de confusion - XGBoost")
print_confusion_table(ym_te, ym_pred_svm, le.classes_, "Matriz de confusion - SVM")

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# --- Grafico de sintesis: laboratorio (BoT-IoT) vs. honeypot real ---
# Estos valores de "real" (recall en honeypot) se obtuvieron ejecutando
# el notebook datasetpropio.ipynb sobre el tráfico capturado; se copian
# aqui manualmente para generar el grafico comparativo final
modelos = ['SVM', 'Random\nForest', 'XGBoost', 'AE +\nXGBoost']
lab = [94.66, 99.79, 99.73, 95.47]
real = [79.61, 47.55, 45.14, 28.77]

x = np.arange(len(modelos))
width = 0.35

fig, ax = plt.subplots(figsize=(7, 5))
ax.bar(x - width/2, lab, width, label='Accuracy en BoT-IoT (laboratorio)', color='#4C72B0')
ax.bar(x + width/2, real, width, label='Recall en honeypot (real)', color='#DD8452')
ax.set_ylabel('Porcentaje (%)')
ax.set_title('Rendimiento en laboratorio vs. generalizacion real')
ax.set_xticks(x)
ax.set_xticklabels(modelos)
ax.legend()
ax.set_ylim(0, 105)
plt.tight_layout()
plt.savefig('sintesis_lab_vs_real.png', dpi=200, bbox_inches='tight')
plt.show()

**Segundo análisis:**

In [ ]:
import pandas as pd

# --- Analisis comparativo de distribuciones: preparacion del lado BoT-IoT ---
# Recargamos la muestra de BoT-IoT SIN escalar (antes del MinMaxScaler),
# porque queremos comparar las distribuciones originales de cada
# caracteristica frente a las del honeypot, no sus versiones normalizadas
features = ['pkts','bytes','spkts','dpkts','sbytes','dbytes','rate','srate','drate']

df = pd.read_csv('bot_iot_sample_ml.csv', low_memory=False)
df_botiot = df[features].copy()

print(f"BoT-IoT cargado: {len(df_botiot):,} filas")
print(df_botiot.describe().round(3))

In [ ]:
from google.colab import files

# Subimos manualmente el features.csv generado por el honeypot (el
# resultado del script zeek_to_botiot.py sobre el conn.log de Zeek)
uploaded = files.upload()

In [ ]:
# Cargamos el CSV del honeypot y nos quedamos solo con las 9 columnas
# comparables con BoT-IoT, sin escalar (misma logica que la celda anterior)
df_honeypot = pd.read_csv('features2.csv')
df_hp = df_honeypot[features].copy()

print(f"Honeypot cargado: {len(df_hp):,} filas")
print(df_hp.describe().round(3))

In [ ]:
import numpy as np
from scipy import stats
import matplotlib.pyplot as plt

# --- Comparacion estadistica BoT-IoT vs. honeypot (sin normalizar) ---

# 1. Estadisticas descriptivas comparadas (media, mediana, desviacion
#    tipica) para cada una de las 9 caracteristicas, en ambas fuentes
comparison_rows = []
for f in features:
    b, h = df_botiot[f], df_hp[f]
    comparison_rows.append({
        'feature': f,
        'botiot_mean': b.mean(), 'botiot_median': b.median(), 'botiot_std': b.std(),
        'hp_mean': h.mean(), 'hp_median': h.median(), 'hp_std': h.std(),
    })
comparison_df = pd.DataFrame(comparison_rows)
print(comparison_df.to_string(index=False))

# 2. Test de Kolmogorov-Smirnov: cuantifica la divergencia maxima entre
#    las funciones de distribucion acumulada de ambas fuentes (0 =
#    distribuciones identicas, 1 = completamente disjuntas)
print("\n=== Test de Kolmogorov-Smirnov (BoT-IoT vs honeypot) ===")
for f in features:
    stat, pvalue = stats.ks_2samp(df_botiot[f], df_hp[f])
    print(f"{f:10s}  KS-stat={stat:.4f}  p-value={pvalue:.2e}")

# 3. Boxplots comparativos en escala logaritmica (log(1+x)) para poder
#    visualizar variables con rangos muy dispares en un mismo grafico
fig, axes = plt.subplots(3, 3, figsize=(15, 12))
for i, f in enumerate(features):
    ax = axes[i // 3, i % 3]
    data = [np.log1p(df_botiot[f].clip(lower=0)), np.log1p(df_hp[f].clip(lower=0))]
    ax.boxplot(data, labels=['BoT-IoT', 'Honeypot'], showfliers=False)
    ax.set_title(f)
    ax.set_ylabel('log(1+x)')
plt.tight_layout()
plt.savefig('comparacion_distribuciones.png', dpi=200, bbox_inches='tight')
plt.show()